# Setup

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not ((repo_root / "Final").exists() and (repo_root / "Sprint 3").exists()):
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root.")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
from datetime import datetime
from typing import Any
import pandas as pd

from Final.config import default_config
from Final.paths import FINAL_ROOT
from Final.shared_utils import setup_logging, get_logger

from Final.models import (
    ExperimentState,
)
from Final.gating import (
    evaluate_module_card,
    decide_module_status,
    module_cards_to_frame,
)
from Final.labeling.pipeline import LabelingPipeline, LabelingPipelineConfig

In [ ]:
cfg = default_config()

logger = setup_logging(
    name="shrub",
    log_dir=cfg.output.logs_root,
    log_filename="main_pipeline.log",
    force=True,
)

logger.info("Initialized main pipeline notebook.")
logger.info("FINAL_ROOT = %s", FINAL_ROOT)

MAIN_OUTPUT_ROOT = cfg.output.root / "main"
MAIN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_ROOT = MAIN_OUTPUT_ROOT

MAIN_MANIFEST_DIR = MAIN_OUTPUT_ROOT / "manifests"
MAIN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS_ROOT = MAIN_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

2026-04-10 04:38:21 | INFO     | shrub | Initialized main pipeline notebook.
2026-04-10 04:38:21 | INFO     | shrub | FINAL_ROOT = /home/jovyan/work/Dry-shRub/shrub/Final


# Controller

In [ ]:
@dataclass
class TrialRecord:
    trial_id: str
    created_at: str
    status: str = "initialized"   # initialized / running / partial / completed / failed

    labeling_config: dict[str, Any] = field(default_factory=dict)
    features_config: dict[str, Any] = field(default_factory=dict)
    modeling_config: dict[str, Any] = field(default_factory=dict)
    postprocessing_config: dict[str, Any] = field(default_factory=dict)

    labeling_result: dict[str, Any] = field(default_factory=dict)
    features_result: dict[str, Any] = field(default_factory=dict)
    modeling_result: dict[str, Any] = field(default_factory=dict)
    postprocessing_result: dict[str, Any] = field(default_factory=dict)

    qa_summary: dict[str, Any] = field(default_factory=dict)
    score_summary: dict[str, Any] = field(default_factory=dict)

    notes: list[str] = field(default_factory=list)


@dataclass
class ExperimentRegistry:
    experiment_name: str
    created_at: str
    updated_at: str
    trials: list[dict[str, Any]] = field(default_factory=list)
    best_trial_id: str | None = None
    best_score: float | None = None

In [ ]:
def now_iso() -> str:
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def experiment_dir(experiment_name: str) -> Path:
    d = EXPERIMENTS_ROOT / experiment_name
    d.mkdir(parents=True, exist_ok=True)
    return d


def registry_path(experiment_name: str) -> Path:
    return experiment_dir(experiment_name) / "registry.json"


def trials_dir(experiment_name: str) -> Path:
    d = experiment_dir(experiment_name) / "trials"
    d.mkdir(parents=True, exist_ok=True)
    return d


def trial_path(experiment_name: str, trial_id: str) -> Path:
    return trials_dir(experiment_name) / f"{trial_id}.json"


def save_registry(registry: ExperimentRegistry) -> None:
    registry.updated_at = now_iso()
    registry_path(registry.experiment_name).write_text(
        json.dumps(asdict(registry), indent=2, default=str),
        encoding="utf-8",
    )


def load_registry(experiment_name: str) -> ExperimentRegistry | None:
    path = registry_path(experiment_name)
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    return ExperimentRegistry(**payload)


def save_trial(experiment_name: str, trial: TrialRecord) -> None:
    trial_path(experiment_name, trial.trial_id).write_text(
        json.dumps(asdict(trial), indent=2, default=str),
        encoding="utf-8",
    )


def load_trial(experiment_name: str, trial_id: str) -> TrialRecord:
    payload = json.loads(trial_path(experiment_name, trial_id).read_text(encoding="utf-8"))
    return TrialRecord(**payload)


def new_trial_id(registry: ExperimentRegistry) -> str:
    return f"trial_{len(registry.trials) + 1:03d}"

In [ ]:
# Initialize Experiment

EXPERIMENT_NAME = "shrub_map"

registry = load_registry(EXPERIMENT_NAME)
if registry is None:
    registry = ExperimentRegistry(
        experiment_name=EXPERIMENT_NAME,
        created_at=now_iso(),
        updated_at=now_iso(),
        trials=[],
        best_trial_id=None,
        best_score=None,
    )
    save_registry(registry)
    logger.info("Created new experiment registry: %s", EXPERIMENT_NAME)
else:
    logger.info("Loaded existing experiment registry: %s", EXPERIMENT_NAME)

registry

In [ ]:
# Create new trial or resume existing one

RESUME_TRIAL_ID = None   # set like "trial_003" to resume

if RESUME_TRIAL_ID is not None:
    trial = load_trial(EXPERIMENT_NAME, RESUME_TRIAL_ID)
    logger.info("Resumed trial %s", trial.trial_id)
else:
    trial = TrialRecord(
        trial_id=new_trial_id(registry),
        created_at=now_iso(),
        status="initialized",
        notes=[],
    )
    save_trial(EXPERIMENT_NAME, trial)
    logger.info("Initialized new trial %s", trial.trial_id)

trial

In [ ]:
state = ExperimentState(
    experiment_name=f"{EXPERIMENT_NAME}:{trial.trial_id}",
    active_modules=[],
)

state.section_status = {
    "labeling": "not_run",
    "features": "not_run",
    "modeling": "not_run",
    "postprocessing": "not_run",
}

state

## Labeling

In [ ]:
labeling_cfg = LabelingPipelineConfig(
    sprint3_variant="revised",
    sprint3_variants=("original", "revised"),
    run_sprint3=True,
    max_ptx_per_site=1,
    force_rerun_sprint3=False,
    require_success_artifacts_sprint3=True,
    cleanup_ptx_after_all_variants=True,
    cleanup_stale_ptx_before_run=True,
    stale_ptx_days=2,
    use_shape_descriptors=True,
    use_temporal_confidence=False,
    use_boundary_confidence=True,
    use_transform_confidence=False,
    use_object_subspace_filter=False,
    rasterization_mode="circle",
    multires=cfg.raster.create_multires,
    force_rerun_sprint4=False,
    force_refresh_site_assets=False,
    nonfatal_qa_overlay=True,
)

trial.labeling_config = asdict(labeling_cfg)
trial.status = "running"
save_trial(EXPERIMENT_NAME, trial)

trial.labeling_config

In [ ]:
labeling_pipeline = LabelingPipeline(
    cfg,
    pipeline_config=labeling_cfg,
)

labeling_result = labeling_pipeline.run()

print("Sprint 3 manifest:", labeling_pipeline.sprint3_manifest_csv)
print("Sprint 4 manifest:", labeling_pipeline.sprint4_manifest_csv)
print("PTX cache root:", labeling_pipeline.ptx_cache_root)

In [ ]:
# Update State based on modules


In [ ]:
trial.labeling_result = {
    "success": labeling_result.success,
    "status": labeling_result.status,
    "metrics": labeling_result.metrics,
    "qa_outputs": labeling_result.qa_outputs,
    "notes": labeling_result.notes,
}

state.section_status["labeling"] = labeling_result.status
state.raster_outputs = labeling_result.raster_outputs
state.object_outputs = labeling_result.object_outputs
state.qa_outputs["labeling"] = labeling_result.qa_outputs

save_trial(EXPERIMENT_NAME, trial)

print("Labeling section status:", state.section_status["labeling"])
print("Labeling metrics:", labeling_result.metrics)

trial.labeling_result

In [ ]:
evaluated_modules = {}
for name, card in MODULE_REGISTRY.items():
    evaluated = evaluate_module_card(card)
    evaluated.status = decide_module_status(evaluated)
    evaluated_modules[name] = evaluated

module_summary_df = module_cards_to_frame(evaluated_modules)
display(module_summary_df)

## Features

In [ ]:
features_cfg = {
    "enabled": False,
    "feature_families": [],
    "notes": "Placeholder only for now.",
}

trial.features_config = features_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.features_config

In [ ]:
trial.features_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Features pipeline not implemented yet."],
}

state.section_status["features"] = "placeholder_not_run"
state.qa_outputs["features"] = trial.features_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.features_result

# Modeling

In [ ]:
modeling_cfg = {
    "enabled": False,
    "model_family": None,
    "notes": "Placeholder only for now.",
}

trial.modeling_config = modeling_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.modeling_config

In [ ]:
trial.modeling_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Modeling pipeline not implemented yet."],
}

state.section_status["modeling"] = "placeholder_not_run"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.modeling_result

# Post-processing

In [ ]:
postprocessing_cfg = {
    "enabled": False,
    "steps": [],
    "notes": "Placeholder only for now.",
}

trial.postprocessing_config = postprocessing_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.postprocessing_config

In [ ]:
trial.postprocessing_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Postprocessing pipeline not implemented yet."],
}

state.section_status["postprocessing"] = "placeholder_not_run"
state.qa_outputs["postprocessing"] = trial.postprocessing_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.postprocessing_result

# Finalize Trial

In [ ]:
trial.qa_summary = {
    "integrity": {
        "labeling": trial.labeling_result.get("status"),
        "features": trial.features_result.get("status"),
        "modeling": trial.modeling_result.get("status"),
        "postprocessing": trial.postprocessing_result.get("status"),
    },
    "section_level": {
        "labeling": "placeholder",
        "features": "placeholder",
        "modeling": "placeholder",
        "postprocessing": "placeholder",
    },
    "cross_section": {
        "label_to_model_feedback": "placeholder",
        "feature_to_model_feedback": "placeholder",
        "end_to_end_feedback": "placeholder",
    },
}

save_trial(EXPERIMENT_NAME, trial)
trial.qa_summary

In [ ]:
labeling_object_rows = trial.labeling_result.get("metrics", {}).get("n_object_rows", 0)
labeling_artifact_rows = trial.labeling_result.get("metrics", {}).get("n_artifact_rows", 0)

trial.score_summary = {
    "labeling_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
    "features_score_placeholder": None,
    "modeling_score_placeholder": None,
    "postprocessing_score_placeholder": None,
    "composite_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
}

save_trial(EXPERIMENT_NAME, trial)
trial.score_summary

In [ ]:
trial.status = "partial" if (
    trial.features_result.get("status") == "placeholder_not_run"
    or trial.modeling_result.get("status") == "placeholder_not_run"
    or trial.postprocessing_result.get("status") == "placeholder_not_run"
) else "completed"

save_trial(EXPERIMENT_NAME, trial)

trial

In [ ]:
trial_record_min = {
    "trial_id": trial.trial_id,
    "created_at": trial.created_at,
    "status": trial.status,
    "labeling_config": trial.labeling_config,
    "features_config": trial.features_config,
    "modeling_config": trial.modeling_config,
    "postprocessing_config": trial.postprocessing_config,
    "score_summary": trial.score_summary,
}

registry.trials = [t for t in registry.trials if t["trial_id"] != trial.trial_id]
registry.trials.append(trial_record_min)

score = trial.score_summary.get("composite_score_placeholder")
if score is not None:
    if registry.best_score is None or score > registry.best_score:
        registry.best_score = score
        registry.best_trial_id = trial.trial_id

save_registry(registry)

registry

In [ ]:
trials_df = pd.DataFrame(registry.trials)

if not trials_df.empty:
    if "score_summary" in trials_df.columns:
        trials_df["composite_score_placeholder"] = trials_df["score_summary"].apply(
            lambda x: x.get("composite_score_placeholder") if isinstance(x, dict) else None
        )

    display(
        trials_df[
            ["trial_id", "created_at", "status", "composite_score_placeholder"]
        ].sort_values("trial_id")
    )

    print("Best trial:", registry.best_trial_id)
    print("Best score:", registry.best_score)
else:
    print("No trials recorded yet.")

In [ ]:
INSPECT_TRIAL_ID = trial.trial_id  # change manually

inspect_trial = load_trial(EXPERIMENT_NAME, INSPECT_TRIAL_ID)
inspect_trial